# Week 1 - Tuesday Labs

### Verifying OpenAI Key


In [3]:
# verify_key.py

import os
from dotenv import load_dotenv

# Load .env file into environment variables
load_dotenv()

# Retrieve the key
api_key = os.getenv("OPENAI_API_KEY")
open_router_api_key = os.getenv("OPENROUTER_API_KEY")

# Validating the OpenAI API key
if not api_key:
    print("ERROR: OPENAI_API_KEY not found in .env file")
elif not api_key.startswith("sk-"):
    print("WARNING: Key does not look like an OpenAI key (should start with 'sk-')")
else:
    masked = api_key[:7] + "..." + api_key[-4:]
    print(f"Key loaded successfully: {masked}")
    print(f"Length: {len(api_key)} characters")

# Validating the OpenRouter API key
if not open_router_api_key:
    print("ERROR: OPENROUTER_API_KEY not found in .env file")
elif not open_router_api_key.startswith("sk-"):
    print("WARNING: Key does not look like an OpenRouter key (should start with 'sk-')")
else:
    masked = open_router_api_key[:7] + "..." + open_router_api_key[-4:]
    print(f"Key loaded successfully: {masked}")
    print(f"Length: {len(open_router_api_key)} characters")

Key loaded successfully: sk-proj...ZbUA
Length: 164 characters
Key loaded successfully: sk-or-v...cd7b
Length: 73 characters


### Challenge: Make the Loader More Defensive

Extend Lab 4 so the verifier also checks that the key is not the placeholder ‘sk-your-key-here’ and exits with a helpful message if it is.


In [4]:
# verify_key.py

import os
from dotenv import load_dotenv

# Load .env file into environment variables
load_dotenv()

# Retrieve the key
api_key = os.getenv("OPENAI_API_KEY")
open_router_api_key = os.getenv("OPENROUTER_API_KEY")

# Validating the OpenAI API key
# TODO 1: Exit with sys.exit(1) and a friendly message if api_key is None.
if not api_key:
    print("ERROR: OPENAI_API_KEY not found in .env file")
    exit(1)
# TODO 2: Exit if api_key equals the placeholder 'sk-your-key-here'.
elif api_key == "sk-your-key-here":
    print("ERROR: Please replace the placeholder key with your actual OpenAI API key")
    exit(1)
# TODO 3: Otherwise print 'Key loaded successfully' with the masked key.
elif not api_key.startswith("sk-"):
    print("WARNING: Key does not look like an OpenAI key (should start with 'sk-')")
else:
    masked = api_key[:7] + "..." + api_key[-4:]
    print(f"Key loaded successfully: {masked}")
    print(f"Length: {len(api_key)} characters")

# Validating the OpenRouter API key (repeating the same validation logic for OpenRouter)
# TODO: Repeat the same validation logic for open_router_api_key as done for api_key.
if not open_router_api_key:
    print("ERROR: OPENROUTER_API_KEY not found in .env file")
    exit(1)
elif open_router_api_key == "sk-your-key-here":
    print("ERROR: Please replace the placeholder key with your actual OpenRouter API key")
    exit(1)
elif not open_router_api_key.startswith("sk-"):
    print("WARNING: Key does not look like an OpenRouter key (should start with 'sk-')")
else:
    masked = open_router_api_key[:7] + "..." + open_router_api_key[-4:]
    print(f"Key loaded successfully: {masked}")
    print(f"Length: {len(open_router_api_key)} characters")

Key loaded successfully: sk-proj...ZbUA
Length: 164 characters
Key loaded successfully: sk-or-v...cd7b
Length: 73 characters


## Lab 5: Making Our First OpenAI API Call


In [10]:
# first_api_call.py
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
# When using OpenAI, you can set it up in this way
# client = OpenAI()

# When using OpenRouter, you can set it up in this way
api_key = os.getenv("OPENROUTER_API_KEY")
client = OpenAI(api_key=api_key, base_url="https://openrouter.ai/api/v1")

messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful health assistant for AfyaPlus Health. "
            "Provide general health guidance. Never diagnose conditions "
            "or prescribe medication. Always recommend consulting a "
            "healthcare professional for serious concerns."
        ),
    },
    {
        "role": "user",
        "content": "I have been having headaches for three days. Should I be worried?",
    },
]

try:
    print(f"Making the API Call...")
    response = client.chat.completions.create(
        model="openrouter/free",  # or "gpt-4o-mini" when using ChatGPT
        messages=messages,  # type: ignore
        temperature=0.3,
        max_tokens=300,
    )

    # Writing the response to a file so that we can review the dictionary's keys
    with open(os.path.join(os.getcwd(), "first_api_call_response.json"), "w") as f:
        json.dump(response.model_dump(), f, indent=4)

    ai_message = response.choices[0].message.content
    assert ai_message is not None, "AI message content is missing in the response."

    usage = response.usage
    assert usage is not None, "Usage information is missing in the response."

    # Response output
    print("\n\n--- AfyaPlus Health Assistant ---")
    print("Patient: I have been having headaches for three days. Should I be worried?")
    print(f"Assistant: {ai_message}")

    print("\n\n--- Usage Statistics ---")
    print(f"Prompt tokens: {usage.prompt_tokens}")
    print(f"Response tokens: {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")

except Exception as e:
    print(f"Error calling the API: {e}")
    print("Check your API key, model ID, credits, and network connection.")

Making the API Call...


--- AfyaPlus Health Assistant ---
Patient: I have been having headaches for three days. Should I be worried?
Assistant: I'm sorry to hear you've been dealing with headaches for three days. While I can't diagnose or provide medical advice, I can offer some general guidance.

**When to consider seeing a healthcare professional:**
- Headaches that are severe or worsening
- Headaches accompanied by fever, stiff neck, confusion, seizures, or neurological symptoms
- Headaches that started suddenly and are "the worst headache of your life"
- Headaches following an injury to the head or neck
- New headaches that begin after age 50
- Changes in your usual headache pattern

**For common, mild headaches:**
- Rest and relaxation
- Staying hydrated
- Over-the-counter pain relievers (following package directions)
- Applying a cold or warm compress to your head or neck
- Getting adequate sleep

Since your headaches have persisted for three days, it would be a good idea to con

### Challenge: Cost Calculator

The CTO wants to know how much it will cost to run AfyaPlus with GPT-4o-mini. Building on Lab 5, write a function that estimates monthly API costs based on average tokens per message, daily message volume, and OpenAI’s pricing. GPT-4o-mini pricing: $0.15 per 1M input tokens, $0.60 per 1M output tokens.


In [14]:
# cost_calculator.py (starter)
def estimate_monthly_cost(
    avg_input_tokens,
    avg_output_tokens,
    daily_messages,
    days_per_month=30,
):
    """
    Calculate estimated monthly API costs for AfyaPlus.
    GPT-4o-mini pricing:
    - Input:  $0.15 per 1,000,000 tokens
    - Output: $0.60 per 1,000,000 tokens
    """
    # YOUR CODE HERE
    monthly_input_cost = (avg_input_tokens * daily_messages * days_per_month) / (
        0.15 * 1_000_000
    )
    monthly_output_cost = (avg_output_tokens * daily_messages * days_per_month) / (
        0.60 * 1_000_000
    )
    total_monthly_cost = monthly_input_cost + monthly_output_cost
    monthly_messages = daily_messages * days_per_month
    cost_per_message = (
        total_monthly_cost / monthly_messages if monthly_messages > 0 else 0
    )

    return {
        "monthly_input_cost": monthly_input_cost,
        "monthly_output_cost": monthly_output_cost,
        "total_monthly_cost": total_monthly_cost,
        "monthly_messages": daily_messages * days_per_month,
        "cost_per_message": cost_per_message,
    }


# Test: 5,000 messages/day, avg 100 input + 150 output tokens
result = estimate_monthly_cost(
    avg_input_tokens=100,
    avg_output_tokens=150,
    daily_messages=5000,
)
print(f"Monthly cost estimate: ${result['total_monthly_cost']:.2f}")
print(f"Monthly messages: {result['monthly_messages']:.2f}")
print(f"Input cost:  ${result['monthly_input_cost']:.2f}")
print(f"Output cost: ${result['monthly_output_cost']:.2f}")
print(f"Total monthly: ${result['total_monthly_cost']:.2f}")
print(f"Cost per message: ${result['cost_per_message']:.6f}")

Monthly cost estimate: $137.50
Monthly messages: 150000.00
Input cost:  $100.00
Output cost: $37.50
Total monthly: $137.50
Cost per message: $0.000917


# Running AI Locally with Ollama

### Install Ollama on your device

1. Install Ollama (visit https://ollama.ai/download for your OS)

- macOS: brew install ollama
- Linux: curl -fsSL https://ollama.ai/install.sh | sh
- Windows: Download installer from https://ollama.ai/download

2. You'll need to start the ollama service before running this notebook.
   You can do this by running the following commands in a separate terminal:

```bash
ollama serve
```

3. Then, you can pull the llama3.2 model in a separate terminal and add it to your environment:

```bash
ollama pull llama3.2
```

4. Once it's done downloading, verify it works by running the following command:

```bash
ollama run llama3.2 "Hello, are you working?"
```

5. To add the ollama package to your environment, you can run:

```bash
pip install ollama
```

Or, if you're using uv like me, you can run:

```bash
uv add ollama
```


In [19]:
from openai import OpenAI

# Connect to local Ollama instance (no API key needed)
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful health assistant for AfyaPlus Health. "
            "Provide general health guidance. Never diagnose conditions "
            "or prescribe medication."
        ),
    },
    {
        "role": "user",
        "content": "I have been having headaches for three days. Should I be worried?",
    },
]

response = client.chat.completions.create(
    model="llama3.2", messages=messages, temperature=0.3  # type: ignore
)

# Writing the response to a file so that we can review the dictionary's keys
with open(os.path.join(os.getcwd(), "ollama_api_call_response.json"), "w") as f:
    json.dump(response.model_dump(), f, indent=4)

print("--- Local Model Response ---")
print(response.choices[0].message.content)

--- Local Model Response ---
I'm not a medical professional, but I can offer some general guidance. If you're experiencing frequent or severe headaches, it's always a good idea to consult with a healthcare professional for personalized advice.

That being said, if you've been having headaches for three days, it's possible that they could be caused by a variety of factors, such as:

1. Stress or fatigue
2. Dehydration or lack of sleep
3. Eye strain or poor posture
4. Sinus pressure or allergies
5. Tension or muscle strain

In most cases, headaches caused by these factors can be managed with self-care measures such as:

1. Staying hydrated by drinking plenty of water
2. Getting enough rest and practicing relaxation techniques, like deep breathing or meditation
3. Applying cold or warm compresses to the forehead or neck
4. Taking over-the-counter pain relievers, such as acetaminophen or ibuprofen, as directed

However, if you're experiencing any of the following, you should seek medical a

### Challenge: Cloud vs Local Latency Comparison

Olu needs to present a recommendation to the CTO. Building on Lab 5 (cloud) and Lab 6 (local), write a script that sends the same patient message to both endpoints, times each call, and prints a comparison. Note: this requires Ollama to be running locally.


In [ ]:
# model_comparison.py (starter)
import os
import time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
# I'll be using OpenRouter for the cloud model
cloud_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

# and Ollama for the local model in this example.
local_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

SYSTEM_PROMPT = "You are a health assistant. Provide brief, safe guidance."
patient_message = "I have chest pain when I breathe deeply"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": patient_message},
]

# TODO 1: time the cloud call and store the response
start_time = time.time()
cloud_response = cloud_client.chat.completions.create(
    model="openrouter/free", messages=messages, temperature=0.3  # type: ignore
)
end_time = time.time()
cloud_time = end_time - start_time

assert (
    cloud_response.choices[0].message.content is not None
), "Cloud model response content is missing."
truncated_cloud_response = cloud_response.choices[0].message.content[:200]
cloud_model_response_length = len(cloud_response.choices[0].message.content)

# TODO 2: time the local call and store the response
start_time = time.time()
local_response = local_client.chat.completions.create(
    model="llama3.2", messages=messages, temperature=0.3  # type: ignore
)
end_time = time.time()
local_time = end_time - start_time
assert (
    local_response.choices[0].message.content is not None
), "Local model response content is missing."
truncated_local_response = local_response.choices[0].message.content[:200]
local_model_response_length = len(local_response.choices[0].message.content)

# TODO 3: print a 3-row comparison: time, response length, first 200 chars
print("\n\n--- Cloud Model Response ---")
print(f"{truncated_cloud_response}...")
print(f"Cloud model time: {cloud_time:.2f} seconds")

print("\n\n--- Local Model Response ---")
print(f"{truncated_local_response}...")
print(f"Local model time: {local_time:.2f} seconds")

print("\n\n--- Time Comparison ---")
print(f"Cloud model time: {cloud_time:.2f} seconds")
print(f"Local model time: {local_time:.2f} seconds")



--- Cloud Model Response ---
Chest pain that occurs with deep breathing can have several possible causes, ranging from benign to more serious. Here are some common possibilities:

**Pleuritic Pain (Most Likely)**
- Pain from the 
Cloud model time: 18.26 seconds


--- Local Model Response ---
I'm not a doctor, but I can provide some general guidance. If you're experiencing chest pain when breathing deeply, it's essential to seek medical attention. Here are some possible explanations and st
Local model time: 13.93 seconds


--- Time Comparison ---
Cloud model time: 18.26 seconds
Local model time: 13.93 seconds


### Further Challenge D: Multi-Language Support

Many AfyaPlus patients communicate in Swahili or in Sheng (English-Swahili mix). Update the system prompt so that the AI detects the patient’s language and responds in the same language. Test with at least two messages: one in English and one in Swahili.


In [26]:
# multi_language.py
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# I'll be using OpenRouter for the cloud model
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

# TODO: Replace this placeholder with a system prompt that instructs
# the model to detect the patient's language and respond in the same language.
SYSTEM_PROMPT = """
You are an AfyaPlus health assistant. Provide brief guidance.
For any patient message, detect the language and respond in the same language.
"""

test_messages = [
    "I have a fever and headache for two days",
    "Nina maumivu ya kichwa kwa siku tatu",  # Swahili: I have a headache for three days
]

for msg in test_messages:
    response = client.chat.completions.create(
        model="openrouter/free",  # or "gpt-4o-mini" when using ChatGPT
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": msg},
        ],
        temperature=0.3,
    )
    print(f"Patient: {msg}")
    print(f"Assistant: {response.choices[0].message.content}\n")

Patient: I have a fever and headache for two days
Assistant: I’m sorry you’re feeling unwell. Try the following:

1. **Rest** – give your body time to recover.  
2. **Stay hydrated** – drink water, clear broth, or an electrolyte drink.  
3. **Fever/headache relief** – you can take acetaminophen (paracetamol) or ibuprofen, following the dosing instructions on the package.  
4. **Monitor** – if the fever rises above 39 °C (102 °F), lasts more than 48 hours, or you develop new symptoms (e.g., stiff neck, rash, vomiting, confusion), seek medical attention promptly.  

If you’re unsure or symptoms worsen, contact your doctor or go to urgent care. Feel better soon!

Patient: Nina maumivu ya kichwa kwa siku tatu
Assistant: Ndiyo maelezo mafupi:

- **Hydrate**: Piga maji kwa kutosha (kutosha 2‑3 litri kwa siku).  
- **Rest**: Hupumzika katika roomi yenye angavu ndogo, jifunze kupumzika vizuri.  
- **Obatala**: Unaweza kutumia dawa ya kutosha kama paracetamol au ibuprofen (kwa kuzingatia viwang

### Further Challenge E: Configurable Base URL

Modify the Lab 5 code so that base URL and model name come from environment variables rather than being hardcoded. The same script should run against either the cloud or Ollama just by changing .env.


In [25]:
# configurable_model.py
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# TODO 1: Load MODEL_BASE_URL from env with a sensible default.
MODEL_BASE_URL = os.getenv("MODEL_BASE_URL", "https://openrouter.ai/api/v1")

# TODO 2: Load MODEL_NAME from env with a sensible default.
MODEL_NAME = os.getenv("MODEL_NAME", "openrouter/free")

# TODO 3: Initialise OpenAI(base_url=..., api_key=...) using the loaded values.
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=MODEL_BASE_URL,
)

# TODO 4: Pass MODEL_NAME into client.chat.completions.create(model=...).
client = client  # This line is just to avoid linter warnings about unused variable
model = MODEL_NAME

response = client.chat.completions.create(
    model=model, messages=[{"role": "user", "content": "Say hello"}]
)
print(response.choices[0].message.content)

Hello there! 👋 How can I help you today?


### Fix the code · debugging challenge

#### Fix the Code: The Key That Never Loads

Symptom: The script crashes with a TypeError: 'NoneType' object is not subscriptable, even though the key is in your .env file.

Run this broken version, read the error, then diagnose it:

```python
import os
from openai import OpenAI

client = OpenAI() # bug: .env never loaded
print("Key ends with:", os.getenv("OPENAI_API_KEY")[-4:])
```

What is missing that causes os.getenv to return None?

1. The key is too short
2. os must be imported twice
3. OpenAI() needs a password
4. load_dotenv() is never called


In [28]:
from os import environ
from openai import OpenAI

# To fix this, we need to load the .env file before creating the OpenAI client.
load_dotenv()  # Ensure this is called before creating the client

client = OpenAI()  # bug: .env never loaded
print("Key ends with:", environ["OPENAI_API_KEY"][-4:])

Key ends with: ZbUA
